In [ ]:
# Parameters (will be overridden by URL query params when running with Voila)
entityId = "default_entity_id"
# Backend base URL comes from environment when running under Voila
# Set CASE_MGMT_BACKEND_URL in the environment; this default is for local dev only.
import os
backendUrl = os.getenv("CASE_MGMT_BACKEND_URL", "http://localhost:3000")
filter = "Last Month"
# Fallback account ID for Benford's Law analysis
benfordAccountId = "cdtrAcct_3a1f3d24fb2046f2a28dc1fa506d6d69"

In [ ]:
try:
    import requests
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    import plotly.io as pio
    from plotly.subplots import make_subplots
    from IPython.display import display, HTML
    import os
    
    # Set default renderer to basic html/notebook
    pio.renderers.default = "notebook"

    # Output styling
    display(HTML("""
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
        .metric-card { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); text-align: center; }
        .metric-value { font-size: 24px; font-weight: bold; color: #111827; }
        .metric-label { font-size: 12px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.05em; }
        .grid-container { display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 20px; }
        .table-container { margin-top: 20px; overflow-x: auto; }
        table { width: 100%; border-collapse: collapse; font-size: 14px; }
        th { text-align: left; padding: 12px; border-bottom: 1px solid #e5e7eb; color: #6B7280; font-weight: 600; }
        td { padding: 12px; border-bottom: 1px solid #e5e7eb; color: #111827; }
        tr:last-child td { border-bottom: none; }
    </style>
    """))
except Exception as e:
    print(f"Import Error: {e}")

In [ ]:
# Fetch Data Helper
def fetch_data(entity_id, backend_url, time_filter):
    url = f"{backend_url}/api/v1/lakehouse/transaction-history/{entity_id}"
    params = {'tenantId': 'DEFAULT'}
    if time_filter:
        params['filter'] = time_filter
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            return None
        raise e
    except Exception as e:
        raise e

# Main Fetch Logic with Fallback
data = None
fallback_id = "cdtrAcct_3a1f3d24fb2046f2a28dc1fa506d6d69"

try:
    # 1. Try requested entityId
    if entityId and entityId != "default_entity_id":
        data = fetch_data(entityId, backendUrl, filter)

    # 2. If Failed/Empty or Default, try fallback
    if not data:
        print(f"Data not found for {entityId}, trying fallback: {fallback_id}")
        data = fetch_data(fallback_id, backendUrl, filter)
        
    if data:
        summary = data.get('summary', {})
        timeline = data.get('timeline', [])
        volume_dist = data.get('volumeDistribution', [])
        cumulative = data.get('cumulative', [])
        recent = data.get('recentTransactions', [])
        
        df_timeline = pd.DataFrame(timeline)
        if not df_timeline.empty:
            df_timeline['date'] = pd.to_datetime(df_timeline['date'])

        # Normalise volume distribution from backend (bucketStart/transactionCount)
        df_volume = pd.DataFrame(volume_dist)
        if not df_volume.empty:
            if 'bucketStart' in df_volume.columns:
                df_volume['date'] = pd.to_datetime(df_volume['bucketStart'])
            elif 'date' in df_volume.columns:
                df_volume['date'] = pd.to_datetime(df_volume['date'])
            else:
                df_volume['date'] = pd.NaT

            # Ensure we have a transaction-count column for charts
            if 'transactionCount' in df_volume.columns:
                df_volume['txCount'] = df_volume['transactionCount']
            elif 'bucket_tx_count' in df_volume.columns:
                df_volume['txCount'] = df_volume['bucket_tx_count']
            else:
                df_volume['txCount'] = 0
            
        df_cumulative = pd.DataFrame(cumulative)
        if not df_cumulative.empty:
            df_cumulative['date'] = pd.to_datetime(df_cumulative['date'])
            
        df_recent = pd.DataFrame(recent)
            
except Exception as e:
    display(HTML(f"<div style='color: red; padding: 20px; border: 1px solid red; border-radius: 5px;'>Error fetching data: {str(e)}</div>"))
    print(f"Backend URL: {backendUrl}")

In [ ]:
if data:
    # Display Metrics
    total_vol = f"${summary.get('totalVolume', 0):,.2f}"
    total_tx = f"{summary.get('totalTransactions', 0):,}"
    alerts = f"{summary.get('alertsTriggered', 0)}"
    investigated = f"{summary.get('investigated', 0)}"

    display(HTML(f"""
    <div class=\"grid-container\">
        <div class=\"metric-card\">
            <div class=\"metric-label\">Total Volume</div>
            <div class=\"metric-value\">{total_vol}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Total Transactions</div>
            <div class=\"metric-value\">{total_tx}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Alerts Triggered</div>
            <div class=\"metric-value\">{alerts}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Investigated</div>
            <div class=\"metric-value\">{investigated}</div>
        </div>
    </div>
    """))

In [ ]:
if data and not df_timeline.empty:
    # Prepare daily aggregates for dual-axis timeline (amount + count)
    df_daily = None
    if 'date' in df_timeline.columns:
        tmp = df_timeline.copy()
        tmp['date_only'] = tmp['date'].dt.date
        df_daily = (
            tmp.groupby('date_only')
               .agg(txCount=('transactionId', 'count'))
               .reset_index()
        )
        df_daily['date'] = pd.to_datetime(df_daily['date_only'])
        df_daily = df_daily[['date', 'txCount']]

    # Create Charts: 3 rows -> Timeline (dual axis), Cumulative, Volume Distribution
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            "Transaction Timeline (Amount & Count)",
            "Cumulative Value (Running Total)",
            "Transaction Volume Distribution (Count)"
        ),
        row_heights=[0.4, 0.3, 0.3],
        specs=[
            [{"secondary_y": True}],  # Row 1: dual y-axis
            [{}],                      # Row 2
            [{}],                      # Row 3
        ],
    )

    # 1. Transaction Timeline (Amount on left Y-axis)
    fig.add_trace(
        go.Scatter(
            x=df_timeline['date'],
            y=df_timeline['amount'],
            mode='markers+lines',
            name='Amount',
            marker=dict(
                size=8,
                color=df_timeline['isAlerted'].map({True: 'red', False: 'blue'}),
                symbol=df_timeline['isAlerted'].map({True: 'circle', False: 'circle'}),
            ),
            hovertemplate="<b>Date:</b> %{x}<br><b>Amount:</b> $%{y:,.2f}<extra></extra>",
        ),
        row=1,
        col=1,
        secondary_y=False,
    )

    # 1b. Transaction Count (Right Y-axis)
    if df_daily is not None and not df_daily.empty:
        fig.add_trace(
            go.Scatter(
                x=df_daily['date'],
                y=df_daily['txCount'],
                mode='lines+markers',
                name='Transaction Count',
                line=dict(color='#F59E0B', width=2, dash='dot'),
                marker=dict(size=6, color='#F59E0B'),
                hovertemplate="<b>Date:</b> %{x}<br><b>Transactions:</b> %{y}<extra></extra>",
            ),
            row=1,
            col=1,
            secondary_y=True,
        )

    # 2. Cumulative Value (Running Total)
    if not df_cumulative.empty:
        fig.add_trace(
            go.Scatter(
                x=df_cumulative['date'],
                y=df_cumulative['cumulativeAmount'],
                fill='tozeroy',
                mode='lines',
                name='Cumulative Value',
                line=dict(color='#10B981'),  # Green
            ),
            row=2,
            col=1,
        )

    # 3. Transaction Volume Distribution (Bar chart of counts)
    if not df_volume.empty and 'txCount' in df_volume.columns:
        fig.add_trace(
            go.Bar(
                x=df_volume['date'],
                y=df_volume['txCount'],
                name='Transaction Count per Bucket',
                marker_color='#6366F1',
            ),
            row=3,
            col=1,
        )

    # Layout & axes styling
    fig.update_layout(
        height=800,
        showlegend=False,
        margin=dict(l=20, r=20, t=40, b=20),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )

    # Grid lines
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#e5e7eb')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#e5e7eb')

    # Axis labels for the dual-axis timeline
    fig.update_yaxes(title_text="Amount ($)", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Transaction Count", row=1, col=1, secondary_y=True)

    fig.show()

In [ ]:
# Fetch Benford's Law Data and Create Analysis Chart
try:
    # Default date range for analysis (last 90 days)
    from datetime import datetime, timedelta
    end_date = datetime.now()
    start_date = end_date - timedelta(days=90)
    
    # Target account: Use entityId if valid, otherwise fallback
    target_account_id = entityId if entityId and entityId != "default_entity_id" else benfordAccountId
    
    benford_url = f"{backendUrl}/api/v1/lakehouse/lake/analytics/benford/account/{target_account_id}"
    benford_params = {
        'tenantId': 'DEFAULT',
        'from': start_date.strftime('%Y-%m-%d'),
        'to': end_date.strftime('%Y-%m-%d')
    }
    
    benford_response = requests.get(benford_url, params=benford_params)
    
    # If explicitly requested entityId fails, try fallback
    if benford_response.status_code == 404 and target_account_id != benfordAccountId:
        print(f"Benford data not found for {target_account_id}, trying fallback: {benfordAccountId}")
        benford_url = f"{backendUrl}/api/v1/lakehouse/lake/analytics/benford/account/{benfordAccountId}"
        benford_response = requests.get(benford_url, params=benford_params)
        
    benford_response.raise_for_status()
    benford_data = benford_response.json()
    
    # Expected Benford's Law distribution (standard percentages)
    expected_benford = []
    actual_benford = []
    digits = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    
    try:
        if isinstance(benford_data, dict) and 'expected' in benford_data and 'actual' in benford_data:
            # Response format: {expected: {"1": 0.301}, actual: {"1": 0}, sampleSize: N}
            exp_dict = benford_data['expected']
            act_dict = benford_data['actual']
            sample_size = float(benford_data.get('sampleSize', 1))
            
            for d in digits:
                d_str = str(d)
                # Expected values are frequencies (0-1), convert to %
                expected_benford.append(float(exp_dict.get(d_str, 0)) * 100)
                # Actual values are counts, convert to % using sampleSize
                count = float(act_dict.get(d_str, 0))
                actual_benford.append((count / sample_size * 100) if sample_size > 0 else 0)
        else:
            # Fallback for older format
            print("Debug: Using legacy parsing fallback")
            expected_benford = [30.1, 17.6, 12.5, 9.7, 7.9, 6.7, 5.8, 5.1, 4.6]
            actual_benford = [0] * 9 # Default
            
    except Exception as parse_error:
        print(f"Debug: Parse error - {parse_error}")
        expected_benford = [30.1, 17.6, 12.5, 9.7, 7.9, 6.7, 5.8, 5.1, 4.6]
        actual_benford = [0] * 9
    
    # Create Benford's Law Analysis Chart
    fig_benford = go.Figure()
    
    fig_benford.add_trace(go.Bar(
        x=digits,
        y=expected_benford,
        name='Expected (Benford)',
        marker_color='#10b981',
        opacity=0.8
    ))
    
    fig_benford.add_trace(go.Bar(
        x=digits,
        y=actual_benford,
        name='Actual (Transactions)',
        marker_color='#3b82f6',
        opacity=0.8
    ))
    
    fig_benford.update_layout(
        title='Benford\'s Law Distribution Analysis',
        xaxis_title='First Digit',
        yaxis_title='Frequency (%)',
        barmode='group',
        height=400,
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=20, r=20, t=60, b=20),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )
    
    fig_benford.update_xaxes(
        showgrid=True, 
        gridwidth=1, 
        gridcolor='#e5e7eb',
        tickmode='linear',
        dtick=1
    )
    fig_benford.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#e5e7eb')
    

    try:
        fig_html = fig_benford.to_html(full_html=False, include_plotlyjs='cdn')
    except Exception as html_err:
      
        print(f"Warning: could not generate standalone html for figure: {html_err}")
        display(HTML("<h3>Benford's Law Distribution Analysis</h3>"))
        display(HTML("<p style='color: #6B7280; font-size: 14px; margin-bottom: 15px;'>Comparing first-digit distribution of transaction amounts against expected Benford's Law distribution</p>"))
        fig_benford.show()
    else:
        details_html = f"""<details open><summary style='font-size:16px; font-weight:600; cursor:pointer;'>Benford's Law Distribution Analysis</summary><div style='margin-top:8px'>{fig_html}<p style='color: #6B7280; font-size: 14px; margin-top:8px;'>Comparing first-digit distribution of transaction amounts against expected Benford's Law distribution</p></div></details>"""
        display(HTML(details_html))
    
    
    deviation = sum(abs(a - e) for a, e in zip(actual_benford, expected_benford)) / len(digits)
    if deviation > 5:
        deviation_color = "#ef4444"  # Red
        deviation_text = "significant variance"
        deviation_note = "This may indicate potential manipulation or structuring."
    elif deviation > 2:
        deviation_color = "#f59e0b"  # Yellow/Orange
        deviation_text = "moderate variance"
        deviation_note = "Some deviation is normal, but worth monitoring."
    else:
        deviation_color = "#10b981"  # Green
        deviation_text = "normal variance"
        deviation_note = "Distribution follows expected Benford's pattern."
    
    display(HTML(f"""
    <div style=\"background: rgba(0,0,0,0.02); border: 1px solid #e5e7eb; border-radius: 8px; padding: 15px; margin: 15px 0;\">\n        <div style=\"display: flex; align-items: center; gap: 10px;\">\n            <div style=\"width: 8px; height: 8px; border-radius: 50%; background: {deviation_color};\"></div>\n            <span style=\"font-weight: 600; color: #374151;\">Analysis: {deviation_text.title()}</span>\n        </div>\n        <p style=\"margin: 8px 0 0 18px; color: #6B7280; font-size: 14px;\">{deviation_note}</p>\n    </div>\n    """))
    
except Exception as e:
    display(HTML(f"""
    <div style='color: #ef4444; padding: 15px; border: 1px solid #ef4444; border-radius: 8px; background: rgba(239, 68, 68, 0.05);'>\n        <strong>Benford's Analysis Error:</strong> {str(e)}\n        <br><small>Using account: {target_account_id if 'target_account_id' in locals() else benfordAccountId}</small>\n    </div>\n    """))

In [ ]:
if data and not df_recent.empty:
    display(HTML("<h3>Recent Transactions</h3>"))

    # Normalise status for display, but keep the real values
    if 'status' in df_recent.columns:
        def _format_status(v):
            # Backend sends this as a list of labels like ["Alert", "Investigated"]
            if isinstance(v, (list, tuple)):
                return ", ".join(str(x) for x in v)
            if v is None or (isinstance(v, float) and pd.isna(v)):
                return ""
            return str(v)

        df_recent['status'] = df_recent['status'].apply(_format_status)

    # Select and rename columns for display
    cols = ['date', 'type', 'counterparty', 'amount', 'currency', 'status']
    # Ensure cols exist
    valid_cols = [c for c in cols if c in df_recent.columns]
    display_df = df_recent[valid_cols].copy()
    
    # Format Amount
    if 'amount' in display_df.columns:
        display_df['amount'] = display_df['amount'].apply(lambda x: f"{x:,.2f}")
        
    # Render HTML table with styling
    html = display_df.to_html(index=False, classes='table')
    display(HTML(f"<div class='table-container'>{html}</div>"))